<a href="https://colab.research.google.com/github/SeoYun-Y/thinfiler-credit-scoring/blob/seoyun%2Fperf-review-0715/prf_feedback_confirm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## 0. 드라이브 마운트 & 데이터 로드

세션 재시작 후 다시 시작. 158개 컬럼 전체는 필요 없고 PERF1~4만 필요해서, 헤더만 먼저 읽어 컬럼명 확인 후 `usecols`로 필요한 컬럼만 가볍게 로드함 (전체 로드보다 훨씬 빠름).

In [2]:
# 1. 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 경로 설정
import pandas as pd

base = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
path = base + '/data/개인CB_씬파일러.csv'

# 3. PERF 컬럼 확인
cols = pd.read_csv(path, nrows=0).columns.tolist()
perf_cols = [c for c in cols if 'PERF' in c.upper()]
print(perf_cols)

# 4. 필요한 컬럼만 로드
usecols = ['PERF1', 'PERF2', 'PERF3', 'PERF4']
df = pd.read_csv(path, usecols=usecols)
print(df.shape)
print(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['PERF1', 'PERF2', 'PERF3', 'PERF4']
(3254230, 4)
   PERF1  PERF2  PERF3  PERF4
0      0      0      0      0
1      0      0      0      0
2      0      0      0      0
3      0      0      0      0
4      0      0      0      0


## 1. PERF1·4가 PERF2에 포함되는지 확인 (강사님 요청 ①)

강사님이 "1,4번이 2번에 거의 포함되는지 확인"하라고 하셔서, PERF1=1·PERF4=1인 사람들이 PERF2=1에도 해당하는 비율을 계산. 반대로 PERF2만의 고유 케이스도 같이 확인.

**결과: PERF1의 99.78%, PERF4의 99.82%가 PERF2에 포함됨. PERF2 채택 근거 확보.**

In [3]:
# PERF1, PERF2, PERF4 각각 양성 건수
perf1_pos = df[df['PERF1'] == 1]
perf2_pos = df[df['PERF2'] == 1]
perf4_pos = df[df['PERF4'] == 1]

print(f"PERF1=1: {len(perf1_pos)}명")
print(f"PERF2=1: {len(perf2_pos)}명")
print(f"PERF4=1: {len(perf4_pos)}명")

# PERF1이 PERF2에 포함되는 비율
perf1_in_perf2 = perf1_pos['PERF2'].eq(1).sum()
print(f"\nPERF1=1 중 PERF2도 1인 비율: {perf1_in_perf2}/{len(perf1_pos)} = {perf1_in_perf2/len(perf1_pos)*100:.2f}%")

# PERF4가 PERF2에 포함되는 비율
perf4_in_perf2 = perf4_pos['PERF2'].eq(1).sum()
print(f"PERF4=1 중 PERF2도 1인 비율: {perf4_in_perf2}/{len(perf4_pos)} = {perf4_in_perf2/len(perf4_pos)*100:.2f}%")

# PERF2 중 PERF1, PERF4 둘 다 아닌 "PERF2만의 고유 케이스"
perf2_only = perf2_pos[(perf2_pos['PERF1'] == 0) & (perf2_pos['PERF4'] == 0)]
print(f"\nPERF2=1이면서 PERF1,PERF4는 모두 0인 케이스: {len(perf2_only)}명 ({len(perf2_only)/len(perf2_pos)*100:.2f}%)")

# 전체 조합 교차표
cross = df.groupby(['PERF1', 'PERF2', 'PERF4']).size().reset_index(name='count')
print("\n=== PERF1 x PERF2 x PERF4 교차표 ===")
print(cross[cross['count'] > 0].sort_values('count', ascending=False))

PERF1=1: 3133명
PERF2=1: 4668명
PERF4=1: 2767명

PERF1=1 중 PERF2도 1인 비율: 3126/3133 = 99.78%
PERF4=1 중 PERF2도 1인 비율: 2762/2767 = 99.82%

PERF2=1이면서 PERF1,PERF4는 모두 0인 케이스: 1523명 (32.63%)

=== PERF1 x PERF2 x PERF4 교차표 ===
   PERF1  PERF2  PERF4    count
0      0      0      0  3249551
7      1      1      1     2743
2      0      1      0     1523
6      1      1      0      383
3      0      1      1       19
4      1      0      0        6
1      0      0      1        4
5      1      0      1        1


## 2. 연도별 씬파일러 ID 추적을 위한 원본 파일 구조 확인 (강사님 요청 ②)

지금 쓰던 사전필터 파일(`개인CB_씬파일러.csv`)은 이미 씬파일러만 모아둔 거라 "누가 씬파일러였다가 빠졌는지" 추적이 안 됨. 그래서 원본 4개년 파일(`201912_개인CB.csv` 등)이 필요함. 이 파일들에 ID 컬럼과 씬파일러 정의 5개 필드가 그대로 있는지 먼저 확인.

In [4]:
# 원본 파일 중 하나(2019년)로 컬럼 구조 확인
raw_path_2019 = base + '/09.개인_CB정보/201912_개인CB.csv'
cols_2019 = pd.read_csv(raw_path_2019, nrows=0).columns.tolist()

# ID로 추정되는 컬럼 찾기
id_candidates = [c for c in cols_2019 if 'SN' in c.upper() or 'ID' in c.upper() or 'JOIN' in c.upper()]
print("ID 후보 컬럼:", id_candidates)

# 씬파일러 정의 5개 필드 존재 확인
thin_def_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
print("정의 필드 존재 여부:", [c in cols_2019 for c in thin_def_cols])

ID 후보 컬럼: ['ID']
정의 필드 존재 여부: [True, True, True, True, True]


## 3. 4개년 원본 파일에서 씬파일러 ID 뽑아서 연도별 비교

4개 연도(2019~2022) 파일을 각각 로드해서 씬파일러 정의(5개 필드=0)를 만족하는 ID를 뽑고, 연도 간 교집합·신규진입을 계산. 메모리 절약을 위해 파일 처리 후 바로 `del` + `gc.collect()`.

**결과: 연도별 비율은 25-27%로 안정적. 연도 간 유지율 93-99%로 매우 높음 → 대부분 같은 사람이 반복 등장.**

In [6]:
import pandas as pd
import gc

base = '/content/drive/MyDrive/멀티캠퍼스 데이터분석_신용평가 데이터'
thin_def_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']
usecols = ['ID'] + thin_def_cols

years = ['201912', '202012', '202112', '202212']
thin_ids = {}

for y in years:
    p = base + f'/09.개인_CB정보/{y}_개인CB.csv'
    d = pd.read_csv(p, usecols=usecols)
    is_thin = (d[thin_def_cols] == 0).all(axis=1)
    thin_ids[y] = set(d.loc[is_thin, 'ID'])
    print(f"{y}: 전체 {len(d)}명 중 씬파일러 {len(thin_ids[y])}명 ({len(thin_ids[y])/len(d)*100:.2f}%)")
    del d
    gc.collect()

print("\n=== 연도별 씬파일러 ID 교집합 ===")
for i in range(len(years)-1):
    y1, y2 = years[i], years[i+1]
    overlap = thin_ids[y1] & thin_ids[y2]
    print(f"{y1} ∩ {y2}: {len(overlap)}명 "
          f"({y1} 기준 {len(overlap)/len(thin_ids[y1])*100:.2f}% 유지, "
          f"{y2} 기준 {len(overlap)/len(thin_ids[y2])*100:.2f}%가 기존자)")

all_years_thin = thin_ids['201912'] & thin_ids['202012'] & thin_ids['202112'] & thin_ids['202212']
print(f"\n4개년 전부 씬파일러 유지: {len(all_years_thin)}명")

print("\n=== 연도별 신규 진입 ===")
for i in range(1, len(years)):
    y_prev, y_curr = years[i-1], years[i]
    new_entries = thin_ids[y_curr] - thin_ids[y_prev]
    print(f"{y_prev}→{y_curr} 신규 진입: {len(new_entries)}명")

201912: 전체 3129036명 중 씬파일러 802882명 (25.66%)
202012: 전체 3129036명 중 씬파일러 823170명 (26.31%)
202112: 전체 3129036명 중 씬파일러 833405명 (26.63%)
202212: 전체 3129036명 중 씬파일러 794773명 (25.40%)

=== 연도별 씬파일러 ID 교집합 ===
201912 ∩ 202012: 794732명 (201912 기준 98.98% 유지, 202012 기준 96.55%가 기존자)
202012 ∩ 202112: 814560명 (202012 기준 98.95% 유지, 202112 기준 97.74%가 기존자)
202112 ∩ 202212: 781488명 (202112 기준 93.77% 유지, 202212 기준 98.33%가 기존자)

4개년 전부 씬파일러 유지: 739943명

=== 연도별 신규 진입 ===
201912→202012 신규 진입: 28438명
202012→202112 신규 진입: 18845명
202112→202212 신규 진입: 13285명


## 4. 사전필터 파일의 ID 중복 여부 직접 확인

3번에서 확인한 "같은 사람 반복 등장" 현상이 지금 쓰고 있는 사전필터 파일(325만 행)에 실제로 얼마나 있는지 정량화. ID까지 포함해서 다시 로드하고 고유 ID 수, 등장 횟수 분포를 확인.

**결과: 고유 ID 860,762명, 이 중 739,943명(86%)이 4개년 전부 등장. 즉 행 수의 74%가 중복.**

In [7]:
usecols2 = ['ID', 'PERF1', 'PERF2', 'PERF3', 'PERF4']
path = base + '/data/개인CB_씬파일러.csv'
df2 = pd.read_csv(path, usecols=usecols2)

print("전체 행 수:", len(df2))
print("고유 ID 수:", df2['ID'].nunique())
print("중복 ID로 인한 초과 행:", len(df2) - df2['ID'].nunique())

# ID별 등장 횟수 분포
id_counts = df2['ID'].value_counts()
print("\n등장 횟수 분포:")
print(id_counts.value_counts().sort_index())

전체 행 수: 3254230
고유 ID 수: 860762
중복 ID로 인한 초과 행: 2393468

등장 횟수 분포:
count
1     21579
2     24841
3     74399
4    739943
Name: count, dtype: int64


## 5. PERF2 양성 건수도 중복 집계되고 있는지 확인

4번에서 발견한 ID 중복이 PERF2=1 건수(4,668건)에도 영향을 주는지 확인. 고유 인원 기준으로 다시 세어봄.

**결과: 4,668건이지만 실제 고유 인원은 3,112명. 1,042명이 여러 연도에 걸쳐 반복 집계됨 → 나중에 train/test 분할 시 leakage 위험 있음.**

In [8]:
# PERF2=1인 케이스가 중복 ID에서 얼마나 나오는지 확인
perf2_pos2 = df2[df2['PERF2'] == 1]
print("PERF2=1 전체 건수(행 기준):", len(perf2_pos2))
print("PERF2=1 고유 ID 수:", perf2_pos2['ID'].nunique())
print("같은 사람이 여러 연도에 걸쳐 PERF2=1로 중복 집계된 건수:", len(perf2_pos2) - perf2_pos2['ID'].nunique())

# 한 사람이 여러 연도에서 PERF2=1을 반복한 경우가 있는지
perf2_id_counts = perf2_pos2['ID'].value_counts()
print("\nPERF2=1인 사람 중 여러 연도에 걸쳐 반복된 사람:", (perf2_id_counts > 1).sum(), "명")
print(perf2_id_counts[perf2_id_counts > 1].head(10))

PERF2=1 전체 건수(행 기준): 4668
PERF2=1 고유 ID 수: 3112
같은 사람이 여러 연도에 걸쳐 PERF2=1로 중복 집계된 건수: 1556

PERF2=1인 사람 중 여러 연도에 걸쳐 반복된 사람: 1042 명
ID
8A4A4722075F7130B38181713E965C20C2B638EDDA43759C21FA7057DE0C4A71    4
D358CA6C3D00141A443E8049E1EEC849A52D1473C0460E4891D9DBE7512F99B9    4
F8CEA1393A8DE031163ABDF400D56C0F860D723854C905E3372D7D240312ED76    4
11107A09CD7C7C3A04478142D2889D01DE41721374F4921EA89A62B4E99CEA98    4
C86504129936FB37D1D77DDF3B9D1694EFB4D6448F894F304679A53BCCD85591    4
41B3036425679F3B1A6E043F96D7649CFF32D4378E2F37C826020813DC18762B    4
E6DA99F0478AF6931A425C70C94EC5B87BEFDADD31BD88BFD5CF4483BFF7B369    4
133E5C8517274092BA201C316FC9AC2C9AA2FA1A6F3E14193F37697C8BC16A28    4
F12C61F47B80768D63B6F350DB349B2DFCC289F70A418E7609B20792ABA74E1D    4
98D964F8DC5415BA0297C7D91C040EAB10D597F22C98D8D06089A9A0CF57571A    4
Name: count, dtype: int64


## 6. 태영님이 언급한 컬럼(No.48, No.111, C1L120004)의 실제 필드명 확인

팀원 문서에 번호로만 적혀있던 컬럼(No.48, No.111)과 C1L120004의 정확한 필드명·정의를 찾기 위해 공식 코드북(`093-117_금융 합성데이터_데이터구성상세.xlsx`)을 로드. 시트가 여러 개라 전체를 dict로 불러옴.

In [9]:
xlsx_path = base + '/093-117_금융 합성데이터_데이터구성상세.xlsx'
codebook = pd.read_excel(xlsx_path, sheet_name=None)
print(codebook.keys())

dict_keys(['1.회원 정보', '2.신용 정보', '3.승인매출 정보', '4.청구입금 정보', '5.잔액 정보', '6.채널 정보', '7.마케팅 정보', '8.성과 정보', '9.개인CB 정보', '10.기업CB 정보', '11.통신카드CB 결합정보', '12.금융상품정보', '별첨1. 개인기업CB 코드정보', '별첨2. 결합CB 코드정보'])


## 6-1. 코드북 구조 확인

'9.개인CB 정보' 시트의 컬럼 구성(No, 필드한글명, 필드영문명, 설명 등)과 앞부분 샘플을 확인해서, 이후 No.48·No.111 행을 정확히 찾아내기 위한 사전 작업.

In [10]:
sheet9 = codebook['9.개인CB 정보']
print(sheet9.columns.tolist())
print(sheet9.head(10))

['No', '필드한글명', '필드영문명', '설명', '코드여부', '유효값', '타입']
   No                         필드한글명      필드영문명  \
0   1                          기준년월       STDT   
1   2                        대체 Key         ID   
2   3                            성별     GENDER   
3   4                            연령   AGE_BAND   
4   5         3개월내(15일)카드총이용금액(미해지)  C1Z001373   
5   6   3개월내(15일)신용카드일시불총이용금액(해지포함)  C1M2B4W03   
6   7  3개월내(15일)신용카드할부총이용금액합계(해지포함)  C1M2B5W03   
7   8          1년내(15일)카드총이용금액(미해지)  C1Z001386   
8   9                   신용카드건수(미해지)  C1M210000   
9  10        신용카드건수(해지포함)(1개월내신규개설)  C1M210001   

                                                  설명 코드여부    유효값    타입  
0                                           데이터 기준년월    N    NaN  Char  
1                     개별시점을 연결하여 분석할 수 있도록 대체 Key 생성    N    NaN  Char  
2                                       1: 남성, 2: 여성    Y  설명 참고  Char  
3  2: 19~29세\n3: 30~39세\n4: 40~49세\n5: 50~59세\n6:...    Y  설명 참고  Char  
4  기준일로부터 3개월내(15일기준) 전체카드(신용카드/

## 6-2. No.48, No.111, C1L120004 필드명 확정

**결과:**
- No.48 → `L10173000` (최초대출개설일자 경과일수, **해지포함**)
- No.111 → `D10110000` (연체건수, **해제포함**)
- C1L120004 → 최초신용카드개설일자 경과일수 (활성카드, 미해지)

기존 씬파일러 정의 5개 필드는 전부 "미해지" 기준이었는데, 이 3개는 해지·해제 이력까지 포함하는 필드라 태영님이 지적한 맹점(카드/대출 있다가 정리한 사람)을 걸러낼 수 있을 것으로 기대.

In [11]:
# No.48, No.111 확인
print(sheet9[sheet9['No'].isin([48, 111])][['No', '필드한글명', '필드영문명', '설명']])

print("\n=== C1L120004 검색 ===")
print(sheet9[sheet9['필드영문명'] == 'C1L120004'][['No', '필드한글명', '필드영문명', '설명']])

      No                   필드한글명      필드영문명  \
47    48  최초대출개설일자로부터의경과일수(해지포함)  L10173000   
110  111              연체건수(해제포함)  D10110000   

                                                    설명  
47   KCB에 등록된 대출개설정보 중 최초 개설된 개인대출정보의 대출개설일자로부터 기준일...  
110                                  연체정보(미해제+해제)의 총건수  

=== C1L120004 검색 ===
    No                         필드한글명      필드영문명  \
22  23  최초신용카드개설일자로부터경과일수(활성카드)(미해지)  C1L120004   

                                                   설명  
22  최근12개월동안 이용내역이 있는 신용카드 +체크카드(전체) 중최초로 등록된 신용카드...  


## 7. 3개 컬럼이 원본 데이터에 실제로 존재하는지 확인

필드명은 확정됐으니, 2022년 원본 파일에 이 3개 컬럼이 실제로 존재하는지 헤더만 확인.

**결과: 3개 다 존재함. 바로 비교 작업으로 진행.**

In [12]:
new_cols = ['C1L120004', 'L10173000', 'D10110000']
thin_def_cols = ['C1M210000', 'C18233003', 'C18233004', 'C18233005', 'L10220000']

# 2022년 파일로 확인 (가장 최근 시점)
p2022 = base + '/09.개인_CB정보/202212_개인CB.csv'
cols_check = pd.read_csv(p2022, nrows=0).columns.tolist()
print("존재 여부:", [c in cols_check for c in new_cols])

존재 여부: [True, True, True]


## 8. 기존 정의 vs 태영님 제안 조건 비교 (2022년 기준)

기존 정의(5개 필드=0)와 태영님이 제안한 조건(3개 컬럼=0, 해지·해제 이력까지 반영)을 둘 다 계산해서, 기존 정의가 잡던 사람 중 새 조건에서 제외되는 비율을 확인.

**결과: 기존 정의 씬파일러(794,773명) 중 58.03%(461,202명)가 새 조건에서는 제외됨 → 절반 이상이 "카드/대출 있다가 해지"한 사람이었을 가능성.**

In [13]:
usecols3 = ['ID'] + thin_def_cols + new_cols
d3 = pd.read_csv(p2022, usecols=usecols3)

# 기존 정의: 5개 필드 전부 0
existing_thin = (d3[thin_def_cols] == 0).all(axis=1)
print("기존 정의 씬파일러:", existing_thin.sum(), f"({existing_thin.mean()*100:.2f}%)")

# 새 조건: L10173000=0, D10110000=0 (해지/해제 포함이라 과거 이력까지 반영)
# C1L120004는 "활성카드" 기준이라 신용카드 미해지 조건(C1M210000=0)과 사실상 겹칠 가능성 높음 - 별도 확인
strict_new = (d3['L10173000'] == 0) & (d3['D10110000'] == 0) & (d3['C1L120004'] == 0)
print("팀원 제안 3개 컬럼=0 조건:", strict_new.sum(), f"({strict_new.mean()*100:.2f}%)")

# 기존 정의를 만족하지만 새 조건은 불만족 (즉 해지/해제 이력이 있어서 제외되어야 할 사람들)
false_positive_in_existing = existing_thin & ~strict_new
print("\n기존 정의는 씬파일러로 잡지만, 새 조건으로는 제외되는 사람:",
      false_positive_in_existing.sum(),
      f"(기존 정의 중 {false_positive_in_existing.sum()/existing_thin.sum()*100:.2f}%)")

# 교집합 (둘 다 만족)
both = existing_thin & strict_new
print("두 정의 모두 만족:", both.sum())

기존 정의 씬파일러: 794773 (25.40%)
팀원 제안 3개 컬럼=0 조건: 362825 (11.60%)

기존 정의는 씬파일러로 잡지만, 새 조건으로는 제외되는 사람: 461202 (기존 정의 중 58.03%)
두 정의 모두 만족: 333571


## 9. 두 정의 간 실제 연체율 차이 검증 (2022년 기준)

8번에서 확인한 차이가 실제 위험도 차이로 이어지는지 확인하기 위해, PERF2를 붙여서 세 그룹(기존정의만 만족/둘다만족/일반고객)의 연체율을 비교.

**결과: 기존정의만 만족 0.090% vs 둘다만족(진짜 무이력) 0.000%(정확히는 매우 낮음) vs 일반고객 0.537%. 태영님 제안이 더 순도 높은 저위험군을 골라낸다는 게 실증됨.**

In [14]:
usecols4 = usecols3 + ['PERF2']
d4 = pd.read_csv(p2022, usecols=usecols4)

existing_thin = (d4[thin_def_cols] == 0).all(axis=1)
strict_new = (d4['L10173000'] == 0) & (d4['D10110000'] == 0) & (d4['C1L120004'] == 0)

group_existing_only = d4[existing_thin & ~strict_new]  # 기존정의만 만족 (해지이력 있음)
group_both = d4[existing_thin & strict_new]  # 둘 다 만족 (진짜 무이력)

print(f"기존정의만 만족(해지이력 있음) {len(group_existing_only)}명 - PERF2=1 연체율: {group_existing_only['PERF2'].mean()*100:.3f}%")
print(f"둘 다 만족(진짜 무이력) {len(group_both)}명 - PERF2=1 연체율: {group_both['PERF2'].mean()*100:.3f}%")

# 참고: 전체 일반 고객군 연체율
general = d4[~existing_thin]
print(f"일반 고객(비씬파일러) {len(general)}명 - PERF2=1 연체율: {general['PERF2'].mean()*100:.3f}%")

기존정의만 만족(해지이력 있음) 461202명 - PERF2=1 연체율: 0.090%
둘 다 만족(진짜 무이력) 333571명 - PERF2=1 연체율: 0.000%
일반 고객(비씬파일러) 2334263명 - PERF2=1 연체율: 0.537%


## 10. 4개년 전체로 확장해서 재검증

9번은 2022년 한 해만 본 거라, 같은 패턴이 4개년 전체에서 일관되게 나타나는지 확인. 연도별로 기존정의/신조건/둘다만족 인원과 각각의 연체율을 정리한 표로 출력.

**결과: 패턴은 일관되지만(둘다만족 그룹 연체율이 항상 가장 낮음), 예상 못 한 추가 패턴 발견 — 모든 그룹에서 연도가 최근일수록 연체율이 급격히 낮아짐(일반고객 2.64%→0.54%). 이게 다음 검증(12번)으로 이어짐.**

In [15]:
years = ['201912', '202012', '202112', '202212']
usecols4 = ['ID'] + thin_def_cols + new_cols + ['PERF2']

results = []

for y in years:
    p = base + f'/09.개인_CB정보/{y}_개인CB.csv'
    d = pd.read_csv(p, usecols=usecols4)

    existing_thin = (d[thin_def_cols] == 0).all(axis=1)
    strict_new = (d['L10173000'] == 0) & (d['D10110000'] == 0) & (d['C1L120004'] == 0)

    group_existing_only = d[existing_thin & ~strict_new]
    group_both = d[existing_thin & strict_new]
    general = d[~existing_thin]

    results.append({
        '연도': y,
        '기존정의_인원': existing_thin.sum(),
        '신조건_인원': strict_new.sum(),
        '둘다만족_인원': group_both.shape[0] if len(group_both) else 0,
        '기존만족_PERF2양성': group_existing_only['PERF2'].sum(),
        '기존만족_연체율(%)': group_existing_only['PERF2'].mean()*100,
        '둘다만족_PERF2양성': group_both['PERF2'].sum(),
        '둘다만족_연체율(%)': group_both['PERF2'].mean()*100 if len(group_both) else None,
        '일반고객_연체율(%)': general['PERF2'].mean()*100,
    })

    del d
    gc.collect()

result_df = pd.DataFrame(results)
print(result_df.to_string())

       연도  기존정의_인원  신조건_인원  둘다만족_인원  기존만족_PERF2양성  기존만족_연체율(%)  둘다만족_PERF2양성  둘다만족_연체율(%)  일반고객_연체율(%)
0  201912   802882  328018   290647          2237     0.436714           248     0.085327     2.636455
1  202012   823170  430883   396766          1167     0.273684            12     0.003024     1.283422
2  202112   833405  299555   279249           582     0.105025             8     0.002865     0.755479
3  202212   794773  362825   333571           413     0.089549             1     0.000300     0.537172


## 11. PERF 변수의 공식 정의 확인

10번에서 발견한 "연도가 최근일수록 연체율이 낮아지는" 현상의 원인을 파악하기 위해, PERF1~4가 정확히 어떻게 정의되는지 코드북에서 확인.

**결과: PERF2는 "신청일로부터 12개월 내 30일 이상 연체경험 여부". 즉 12개월치 미래 관측이 필요한 지표.**

In [16]:
# 8.성과 정보 시트 또는 9.개인CB 정보 시트에서 PERF 관련 정의 찾기
for sheet_name in ['8.성과 정보', '9.개인CB 정보']:
    s = codebook[sheet_name]
    perf_rows = s[s['필드영문명'].astype(str).str.contains('PERF', na=False)]
    if len(perf_rows) > 0:
        print(f"=== {sheet_name} ===")
        print(perf_rows[['No', '필드한글명', '필드영문명', '설명']].to_string())
        print()

=== 9.개인CB 정보 ===
      No                         필드한글명  필드영문명                            설명
153  154  신청일로부터 12개월 내 90일 이상 연체경험 여부  PERF1  신청일로부터 12개월 내 90일 이상 연체경험 여부
154  155  신청일로부터 12개월 내 30일 이상 연체경험 여부  PERF2  신청일로부터 12개월 내 30일 이상 연체경험 여부
155  156      신청일로부터 3개월 내 신규 대출 발생 여부  PERF3      신청일로부터 3개월 내 신규 대출 발생 여부
156  157    신청일로부터 6개월내 60일 이상 연체경험 여부  PERF4    신청일로부터 6개월내 60일 이상 연체경험 여부



## 12. "신청일" 기준이 정확히 뭔지 확인 (우측절단 가설 검증)

PERF 정의에 나오는 "신청일"이 데이터에서 어떤 필드를 가리키는지 확인. 별도 신청일 필드가 있는지, 아니면 STDT(기준년월)가 신청일 역할을 하는지 체크.

**결과: 별도 신청일 필드는 없고 STDT만 존재 → 스냅샷 기준일 자체가 신청일. 2022년 12월 스냅샷은 2023년 12월까지 관측해야 하는데 우리 데이터는 2022년까지만 있음 → 우측절단(관측기간 부족) 의심 확정.**

In [17]:
# 신청일자 관련 필드가 있는지 확인 (PERF 계산 기준일)
app_date_cols = [c for c in sheet9['필드영문명'].astype(str) if 'STDT' in c.upper() or '신청' in str(sheet9[sheet9['필드영문명']==c]['필드한글명'].values)]
print(app_date_cols)

# STDT(기준년월) 자체가 신청일 대용일 가능성 확인
print(sheet9[sheet9['필드영문명']=='STDT'][['필드한글명','설명']])

['STDT', 'PERF1', 'PERF2', 'PERF3', 'PERF4']
  필드한글명        설명
0  기준년월  데이터 기준년월


## 13. 나이(AGE_BAND) 필터 검토 — 씬파일러 vs 일반 연령 분포 (태영님 문제제기 ④)

태영님이 "20대 3할+30대까지 4할"이라고 관찰한 걸 직접 검증. 씬파일러 그룹과 전체(일반) 그룹의 AGE_BAND 분포를 비교.

**결과: 씬파일러 그룹 20대 35.73%(일반 16.91% 대비 2.1배). 30대는 오히려 일반보다 낮음(10.28% vs 18.41%) → 쏠림은 20대에만 있음.**

In [18]:
usecols5 = ['ID', 'AGE_BAND'] + thin_def_cols + ['PERF2']
p2022 = base + '/09.개인_CB정보/202212_개인CB.csv'
d5 = pd.read_csv(p2022, usecols=usecols5)

existing_thin = (d5[thin_def_cols] == 0).all(axis=1)
thin_group = d5[existing_thin]

print("AGE_BAND 코드 정의 (코드북 기준): 2:19~29세, 3:30~39세, 4:40~49세, 5:50~59세, 6:...")
print("\n=== 씬파일러 그룹 AGE_BAND 분포 ===")
print(thin_group['AGE_BAND'].value_counts().sort_index())
print("\n비율(%):")
print((thin_group['AGE_BAND'].value_counts(normalize=True).sort_index() * 100).round(2))

print("\n=== 전체(일반) 그룹 AGE_BAND 분포 비교 ===")
print((d5['AGE_BAND'].value_counts(normalize=True).sort_index() * 100).round(2))

AGE_BAND 코드 정의 (코드북 기준): 2:19~29세, 3:30~39세, 4:40~49세, 5:50~59세, 6:...

=== 씬파일러 그룹 AGE_BAND 분포 ===
AGE_BAND
1        12
2    283944
3     81738
4     87920
5    112164
6    101222
7     82712
8     40740
9      4321
Name: count, dtype: int64

비율(%):
AGE_BAND
1     0.00
2    35.73
3    10.28
4    11.06
5    14.11
6    12.74
7    10.41
8     5.13
9     0.54
Name: proportion, dtype: float64

=== 전체(일반) 그룹 AGE_BAND 분포 비교 ===
AGE_BAND
1     0.00
2    16.91
3    18.41
4    20.39
5    20.87
6    14.21
7     7.02
8     2.03
9     0.16
Name: proportion, dtype: float64


## 14. 나이 필터 적용 시 규모·연체율 변화 확인

AGE_BAND=2(20대)만, 또는 2·3(20·30대)만 필터링했을 때 인원과 PERF2 연체율이 어떻게 바뀌는지 확인.

**결과: 인원은 36~46%로 줄지만 연체율은 거의 그대로(0.0521%→0.0511%) → 나이는 위험도를 가르는 기준이 아니라 "사회초년생" 타겟팅용 정책적 필터로 봐야 함.**

In [19]:
# 안: AGE_BAND=2(19~29세)만 사회초년생 씬파일러로 한정
age_filtered = thin_group[thin_group['AGE_BAND'] == 2]
print(f"기존 정의: {len(thin_group)}명")
print(f"+ AGE_BAND=2(19~29세) 필터: {len(age_filtered)}명 ({len(age_filtered)/len(thin_group)*100:.2f}%)")

print(f"\n기존 정의 PERF2=1: {thin_group['PERF2'].sum()}건 (연체율 {thin_group['PERF2'].mean()*100:.4f}%)")
print(f"나이필터 후 PERF2=1: {age_filtered['PERF2'].sum()}건 (연체율 {age_filtered['PERF2'].mean()*100:.4f}%)")

# 참고: 20대+30대 둘 다 포함할 경우
age_filtered_23 = thin_group[thin_group['AGE_BAND'].isin([2,3])]
print(f"\n+ AGE_BAND=2,3(19~39세) 필터: {len(age_filtered_23)}명 ({len(age_filtered_23)/len(thin_group)*100:.2f}%)")
print(f"나이필터(20+30대) 후 PERF2=1: {age_filtered_23['PERF2'].sum()}건 (연체율 {age_filtered_23['PERF2'].mean()*100:.4f}%)")

기존 정의: 794773명
+ AGE_BAND=2(19~29세) 필터: 283944명 (35.73%)

기존 정의 PERF2=1: 414건 (연체율 0.0521%)
나이필터 후 PERF2=1: 145건 (연체율 0.0511%)

+ AGE_BAND=2,3(19~39세) 필터: 365682명 (46.01%)
나이필터(20+30대) 후 PERF2=1: 201건 (연체율 0.0550%)


PERF3 노출조건 요약 (어제 확인한 내용)
팀원이 문서에서 "2021년 씬파일러 중 2022년 신규대출/카드 발생자의 연체여부"를 따지자고 제안했는데, 이건 사실상 PERF3(3개월 내 신규대출 발생) 기반 노출조건 방식이야. 어제 이미 여러 각도로 검증했고 결론은 구조적으로 불가능:

PERF3=1(신규대출 발생) 자체가 전체 씬파일러 중 단 75명뿐
조건완화, 6개월 대체신호, 4개년 패널 추적(졸업자 51,917명 찾음)까지 시도했지만 → 졸업자 중 PERF4=1인 사람이 1명뿐
원본파일 조건완화 시 PERF3는 41배(2,191명)로 늘었지만 그래도 PERF4=0
원인: 씬파일러 정의가 "카드/대출 0"인 사람들인데, 이들이 실제로 신규 대출/카드를 발급받는 사건 자체가 극히 드묾. 데이터 자체가 부족한 게 아니라 현상 자체가 희귀한 것

팀원이 문서에서 스스로 "데이터가 너무 적을 것 같다"고 예상한 게 맞았고, 실제로는 그 예상보다 더 심각해서 아예 타겟 구축이 불가능한 수준이었어. → 결론: PERF2가 최선의 대안(이미 채택한 것)이라는 근거가 하나 더 보태진 셈.

## 15. 통신비·학자금 등 카드 외 연체 경로 관련 필드 탐색 (강사님 질문 대응)

강사님이 "씬파일러인데 왜 PERF가 잡히나"라며 통신비·학자금 대출 가능성을 언급하셨어서, 코드북에서 관련 키워드로 필드를 검색.

**결과: 학자금 관련 필드는 없음. 휴대폰번호 이력(AL012G019) 하나만 발견 — 이미 노출편향 검증에 썼던 변수.**

In [20]:
sheet9_all = codebook['9.개인CB 정보']
keyword_cols = sheet9_all[sheet9_all['필드한글명'].astype(str).str.contains('학자금|통신|휴대폰|공공요금', na=False)]
print(keyword_cols[['No','필드한글명','필드영문명','설명']].to_string())

      No              필드한글명      필드영문명                                        설명
142  143  3년내휴대폰번호이력건수(구간화)  AL012G019  KCB에 등록된 고객 신상정보 중 3년 내 휴대폰번호 이력건수 (구간화)


## 16. 연체 발생 씬파일러 vs 정상 씬파일러 간접 프로파일링

직접적인 원인 필드가 없어서, PERF2=1인 씬파일러(연체군)와 PERF2=0인 씬파일러(정상군)의 나이·휴대폰 변경이력을 비교하는 간접적 방법으로 접근.

**결과: 20대는 두 그룹 거의 동일하나 50대에서 연체 비중이 뚜렷하게 높음(22.71% vs 14.11%). 70·80대는 반대로 정상군이 높음(고령층=저위험). 휴대폰 변경이력은 연체군이 1.5배 높음(2.43 vs 1.57) → 라이프스타일 안정성 지표가 씬파일러 내부에서도 유효.**

---

**추가 확인 — 8~10번 결과를 다시 엮어서 본 직접적인 답**

8~10번에서 씬파일러를 A그룹(기존정의만 만족, 과거 카드·대출 있다가 해지/해제한 이력 있음)과 B그룹(둘 다 만족, 진짜 무이력)으로 나눴었는데, 이 두 그룹의 연도별 PERF2 양성 건수를 다시 보면:

| 연도 | A그룹(해지이력) PERF2양성 | B그룹(진짜무이력) PERF2양성 |
|---|---|---|
| 2019 | 2,237 | 248 |
| 2020 | 1,167 | 12 |
| 2021 | 582 | 8 |
| 2022 | 413 | 1 |

2022년 기준 씬파일러 전체 PERF2=1(414건) 중 **413건(99.76%)이 A그룹에서 발생**했고, 진짜 무이력자(B그룹)에서는 단 1건뿐. 다른 연도도 A그룹 쏠림이 뚜렷함(2019년 2,237/2,485 ≈ 90%).

**→ 강사님 질문("씬파일러인데 왜 PERF가 잡히나")에 대한 실질적인 답: 연체는 대부분 진짜 무이력자가 아니라, 과거에 카드·대출을 갖고 있다가 정리한 씬파일러에서 발생함. "과거 신용활동 이력 유무"가 씬파일러 내부 연체 위험을 가르는 가장 뚜렷한 요인으로 확인됨.**

---

**B그룹(진짜 무이력자)에서도 연체가 0이 아닌 이유 — 두 가지로 해석**

B그룹도 연도별로 248→12→8→1로 급감하는데, 이건 두 가지가 섞인 결과로 보임:

1. **우측절단 효과**: 11~12번에서 확인한 관측기간 부족 문제가 표본이 작은 B그룹에서 더 극단적으로 나타난 것. 2019년의 248건이 상대적으로 "제대로 관측된" 값이고, 2022년의 1건은 관측 부족으로 인위적으로 눌린 값일 가능성이 큼

2. **카드·대출 외 경로의 실제 존재 가능성**: B그룹은 "카드·대출 이력이 전혀 없다"는 뜻이지 "모든 금융활동이 없다"는 뜻은 아님. 8개 필드 전부 카드·대출 관련 필드라, 통신비·공과금·소액대출(KCB 미등록) 등 다른 경로의 연체는 애초에 걸러낼 수 없음. B그룹에서도 연체가(극소수나마) 발생한다는 사실 자체가, 강사님이 처음 제기한 "카드 외 연체 경로" 가설을 뒷받침하는 간접 증거가 될 수 있음

**→ 종합하면: 연체의 99%는 과거 신용이력자(A그룹)에서 나오지만, 극소수(B그룹)는 카드·대출 이력이 전혀 없어도 연체가 발생하며 이는 통신비 등 카드 외 경로일 가능성을 시사함. 다만 이 극소수 사례는 우측절단 효과와 섞여있어 완전히 분리해 해석하기는 어려움.**

In [21]:
usecols6 = ['ID'] + thin_def_cols + ['PERF2', 'AGE_BAND', 'AL012G019']
d6 = pd.read_csv(p2022, usecols=usecols6)
existing_thin = (d6[thin_def_cols] == 0).all(axis=1)
thin_group2 = d6[existing_thin].copy()

pos = thin_group2[thin_group2['PERF2'] == 1]
neg = thin_group2[thin_group2['PERF2'] == 0]

print(f"연체 씬파일러: {len(pos)}명 / 정상 씬파일러: {len(neg)}명")
print("\n=== 나이 분포 비교 ===")
print("연체군:\n", pos['AGE_BAND'].value_counts(normalize=True).sort_index()*100)
print("정상군:\n", neg['AGE_BAND'].value_counts(normalize=True).sort_index()*100)

print("\n=== 휴대폰 변경이력(AL012G019) 비교 ===")
print("연체군 평균:", pos['AL012G019'].mean())
print("정상군 평균:", neg['AL012G019'].mean())

연체 씬파일러: 414명 / 정상 씬파일러: 794359명

=== 나이 분포 비교 ===
연체군:
 AGE_BAND
2    35.024155
3    13.526570
4    15.700483
5    22.705314
6     9.178744
7     3.623188
8     0.241546
Name: proportion, dtype: float64
정상군:
 AGE_BAND
1     0.001511
2    35.726794
3    10.282756
4    11.059861
5    14.108231
6    12.737818
7    10.410532
8     5.128538
9     0.543961
Name: proportion, dtype: float64

=== 휴대폰 변경이력(AL012G019) 비교 ===
연체군 평균: 2.42512077294686
정상군 평균: 1.5699035322820034
